# Assignment 3 · Parte 1 – Visualización con Plotly

**Grupo 11** · Diplomado PUCP · Fundamentos de Python

Usamos **Plotly Express**: gráficos interactivos (al pasar el mouse aparece el dato) que se pueden exportar a HTML.

## 1. Cargar los datos

- El archivo está en la carpeta `assignment_3/datos/`, como pide la estructura del trabajo.
- Leemos el dataset.csv del Assignment_2

In [1]:
# Cargamos librerías: pandas para manejar la tabla y plotly.express (px) para graficar
import pandas as pd
import plotly.express as px

# Leemos el dataset del Assignment 2.
# dtype={"ubigeo": str} evita que '01' se convierta en el número 1.
dataset = pd.read_csv("datos/dataset.csv", dtype={"ubigeo": str})

# Texto de la fuente: lo pegamos al pie de cada gráfico (la rúbrica pide citar la fuente)
#FUENTE = "Fuente: dataset.csv del Assignment 2, grupo 11 (declaratorias de emergencia y lluvia en la capital de cada departamento, 1 ene – 31 may 2024)."

- Mostramos las dimensiones y las primeras 5 filas.

In [2]:
# .shape devuelve (filas, columnas): 25 departamentos y 9 variables
dataset.shape

(25, 9)

In [3]:
# Primeras 5 filas para revisar que la tabla se leyó bien
dataset.head(5)

,ubigeo,departamento,declaratorias,prorrogas,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,4,2,Chachapoyas,-6.23169,-77.86903,455.0,0
1,02,Áncash,5,5,Huaraz,-9.52614,-77.52869,1058.3,3
2,03,Apurímac,7,4,Abancay,-13.63426,-72.88380,670.6,2
3,04,Arequipa,7,6,Arequipa,-16.39899,-71.53747,235.3,1
4,05,Ayacucho,7,5,Ayacucho,-13.16380,-74.22345,467.5,1


## 2. Pregunta y columnas

**Temporada analizada:** 1 de enero al 31 de mayo de 2024 (temporada asignada al grupo 11 en el Assignment 2).

**Pregunta:** ¿Los departamentos donde más llovió son también los que recibieron más declaratorias de emergencia?

| Columna | Qué mide |
|---|---|
| `departamento` | nombre del departamento (25) |
| `declaratorias` | número de declaratorias de estado de emergencia |
| `prorrogas` | número de prórrogas de esas declaratorias |
| `lluvia_total_mm` | lluvia acumulada del 1 de enero al 31 de mayo de 2024, medida en la capital del departamento |
| `dias_lluvia_fuerte` | número de días, entre el 1 de enero y el 31 de mayo de 2024 (152 días), con lluvia fuerte en la capital |

## 3. Gráficos interactivos con Plotly Express

### Gráfico 1 · Barras: top 10 de declaratorias por departamento

Primero ordenamos la tabla con `sort_values()` para ver el ranking.

In [4]:
# Ordenamos de mayor a menor (ascending=False) y nos quedamos con los 10 primeros (head(10))
dep_declaratorias = dataset.sort_values("declaratorias", ascending=False).head(10)

# Mostramos solo las columnas que nos interesan para este gráfico
dep_declaratorias[["departamento", "declaratorias", "prorrogas", "lluvia_total_mm"]]

,departamento,declaratorias,prorrogas,lluvia_total_mm
2,Apurímac,7,4,670.6
7,Cusco,7,5,601.0
3,Arequipa,7,6,235.3
4,Ayacucho,7,5,467.5
8,Huancavelica,7,4,819.2
14,Lima,6,5,12.2
9,Huánuco,6,3,416.7
1,Áncash,5,5,1058.3
11,Junín,5,1,1276.9
20,Puno,5,2,728.6


In [5]:
fig = px.bar(
    dep_declaratorias,
    x="departamento", y="declaratorias",       # departamento en el eje X, número de declaratorias en el eje Y
    text_auto=True,                            # escribe el valor encima de cada barra
    color="declaratorias",                     # el color depende del número de declaratorias...
    color_continuous_scale="reds",             # ...con una escala de rojos (más oscuro = más declaratorias)
    # <br> parte el título en dos líneas para que entre completo en el cuadro
    title="La sierra sur concentra el máximo de declaratorias de emergencia (7)",
    labels={"departamento": "Departamento",    # nombres claros para los ejes
            "declaratorias": "Declaratorias de emergencia"},
)
fig.update_traces(textposition="outside", cliponaxis=False)   # el número va fuera de la barra y no se corta
fig.update_layout(coloraxis_showscale=False,                  # ocultamos la barra de color: ya se ve el número
                  template="plotly_white",                    # fondo blanco, más limpio
                  title_font_size=16,
                  margin=dict(b=100))                         # espacio abajo para la fuente
# Texto de la fuente: lo pegamos al pie de cada gráfico (la rúbrica pide citar la fuente)
FUENTE = "Fuente: dataset.csv del Assignment 2, grupo 11 (declaratorias de emergencia y lluvia en la capital de cada departamento, 1 ene – 31 may 2024)."
# Agregamos la fuente al pie del gráfico (xref/yref="paper" = posición relativa al cuadro)
fig.add_annotation(text=FUENTE, xref="paper", yref="paper", x=0, y=-0.32,
                   showarrow=False, font=dict(size=10, color="gray"), xanchor="left")
fig.show()

### Gráfico 1b · Barras horizontales: los 10 departamentos con más lluvia

Barras **horizontales** (`orientation="h"`), como en la clase: los nombres se leen sin girar la cabeza.
El color indica cuántas declaratorias tuvo cada uno, así vemos si "más lluvia" coincide con "más declaratorias".

In [6]:
# nlargest(10, ...) toma los 10 departamentos con MÁS lluvia.
# Luego los ordenamos de menor a mayor para que, en barras horizontales, el más lluvioso quede arriba.
dep_lluvia = dataset.nlargest(10, "lluvia_total_mm").sort_values("lluvia_total_mm")

fig = px.bar(
    dep_lluvia,
    x="lluvia_total_mm", y="departamento",
    orientation="h",                           # barras horizontales
    text_auto=",.0f",                          # valor con separador de miles y sin decimales (1,362)
    color="declaratorias", color_continuous_scale="reds",
    title="De los 10 departamentos con más lluvia, solo Huancavelica llegó al máximo de 7 declaratorias",
    labels={"lluvia_total_mm": "Lluvia acumulada en la capital (mm)",
            "departamento": "",                # el nombre del eje sobra: ya se leen los departamentos
            "declaratorias": "Declaratorias"},
)
fig.update_traces(textposition="outside", cliponaxis=False)
# Ampliamos el eje X un 15 % para que el número de la barra más larga no se salga del cuadro
fig.update_xaxes(range=[0, dep_lluvia["lluvia_total_mm"].max() * 1.15])
fig.update_layout(template="plotly_white", title_font_size=16, margin=dict(b=90))
fig.add_annotation(text=FUENTE, xref="paper", yref="paper", x=0, y=-0.22,
                   showarrow=False, font=dict(size=10, color="gray"), xanchor="left")
fig.show()

### Gráfico 2 · Dispersión: lluvia vs. declaratorias

Dos variables continuas, una en cada eje. Cada punto es un departamento: `hover_name` muestra cuál es al pasar el mouse.

Solo departamentos con al menos 1 declaratoria (quedan 21: salen Callao, Lambayeque, Piura y Tumbes).

In [7]:
# Correlación de Pearson: va de -1 a 1. Cerca de 0 = casi no hay relación lineal entre las dos variables.
correlacion = dataset["lluvia_total_mm"].corr(dataset["declaratorias"])
print(f"Correlación entre lluvia y declaratorias (25 departamentos): {correlacion:.2f}")

Correlación entre lluvia y declaratorias (25 departamentos): 0.20


In [8]:
# Correlación de Pearson: va de -1 a 1. Cerca de 0 = casi no hay relación lineal entre las dos variables.
filtro = dataset[dataset["declaratorias"] > 0]          # los 21 departamentos con al menos 1 declaratoria
correlacion = filtro["lluvia_total_mm"].corr(filtro["declaratorias"])
print(f"Correlación entre lluvia y declaratorias (21 departamentos): {correlacion:.2f}")


Correlación entre lluvia y declaratorias (21 departamentos): -0.14


Con los 25 departamentos la correlación es 0.20; sin los 4 que no tuvieron declaratorias, es −0.14. En ambos casos está cerca de 0: no hay relación entre cuánto llovió y cuántas declaratorias hubo. Que el signo cambie al quitar solo 4 departamentos muestra que la relación es muy débil.

In [9]:
fig = px.scatter(
    dataset[(dataset["declaratorias"] > 0)], # Solo departamentos con al menos 1 declaratoria (quedan 21: salen Callao, Lambayeque, Piura y Tumbes)
    x="lluvia_total_mm", y="declaratorias",
    hover_name="departamento",                          # nombre del departamento al pasar el mouse
    hover_data={"capital": True, "prorrogas": True, "dias_lluvia_fuerte": True},
    text="departamento",                                # escribe el nombre al lado de cada punto
    color="dias_lluvia_fuerte",                         # el COLOR indica los días de lluvia fuerte...
    color_continuous_scale="Blues",                     # ...más oscuro = más días
    range_color=[-3, 19],                               # empieza en -3 para que los de 0 días no salgan blancos (invisibles)
    title="Llover más no trajo más declaratorias:<br>Loreto tuvo la mayor lluvia y solo 3 declaratorias",
    labels={"lluvia_total_mm": "Lluvia total en la capital (mm)",
            "declaratorias": "Declaratorias de emergencia",
            "dias_lluvia_fuerte": "Días de<br>lluvia fuerte",
            "capital": "Capital", "prorrogas": "Prórrogas"},
    height=600,
)
# Todos los puntos del mismo tamaño (14) y con borde gris, así ninguno desaparece
fig.update_traces(marker=dict(size=14, line=dict(width=1, color="DarkSlateGrey")),
                  textposition="top center", textfont_size=10)
# Cajamarca y San Martín están casi en el mismo punto: separamos sus nombres a izquierda y derecha
fig.for_each_trace(lambda t: t.update(textposition=[
    "top left" if d == "Cajamarca" else
    "top right" if d == "San Martín" else
    "top center"
    for d in t.text]))
# Un poco de margen en los ejes para que los puntos y nombres de los bordes no se corten
fig.update_xaxes(range=[-50, dataset["lluvia_total_mm"].max() * 1.08])
fig.update_yaxes(range=[1.5, 7.7])
fig.update_layout(template="plotly_white", title_font_size=16, margin=dict(b=90))
fig.add_annotation(text=FUENTE + " Color = días de lluvia fuerte.",
                   xref="paper", yref="paper", x=0, y=-0.17, showarrow=False,
                   font=dict(size=10, color="gray"), xanchor="left")
fig.show()

### Gráfico 3 · Libre: boxplot por nivel de lluvia

Agrupamos los departamentos en tres niveles de lluvia y comparamos la distribución de declaratorias de cada grupo.
Si la lluvia explicara las emergencias, las cajas deberían subir de izquierda a derecha.

**Por qué usamos `pd.cut` con cortes fijos y no `pd.qcut` (terciles):** con terciles, el corte cae justo en el valor
de Arequipa (235.3 mm) y, por un redondeo mínimo, Arequipa cae en un grupo u otro según la computadora.
Eso cambiaba las medianas y hacía fallar la verificación. Con cortes fijos (200 y 700 mm) el resultado es el mismo
en cualquier computadora.

In [10]:
# pd.cut parte la lluvia en tramos fijos: (0, 200], (200, 700] y (700, 1500]
dataset["nivel_lluvia"] = pd.cut(
    dataset["lluvia_total_mm"],
    bins=[0, 200, 700, 1500],
    labels=["Menos de 200 mm", "200 a 700 mm", "Más de 700 mm"],
)

# Resumen por grupo: cuántos departamentos hay, rango de lluvia y mediana de declaratorias
dataset.groupby("nivel_lluvia", observed=True).agg(
    departamentos=("departamento", "count"),
    lluvia_min=("lluvia_total_mm", "min"),
    lluvia_max=("lluvia_total_mm", "max"),
    mediana_declaratorias=("declaratorias", "median"),
)

,departamentos,lluvia_min,lluvia_max,mediana_declaratorias
nivel_lluvia,,,,
Menos de 200 mm,8,12.2,139.4,3.0
200 a 700 mm,9,235.3,681.7,6.0
Más de 700 mm,8,728.6,1361.8,4.5


In [11]:
orden = ["Menos de 200 mm", "200 a 700 mm", "Más de 700 mm"]

fig3 = px.box(
    dataset,
    x="nivel_lluvia", y="declaratorias",
    color="nivel_lluvia",
    category_orders={"nivel_lluvia": orden},   # fija el orden de las cajas: de menos a más lluvia
    color_discrete_sequence=["#aed6f1", "#5dade2", "#1b4f72"],   # celeste claro → azul oscuro
    points="all",                              # muestra también cada departamento como un punto
    hover_name="departamento",
    title= "Los departamentos con 200 a 700 mm de lluvia recibieron más declaratorias que los que superaron los 700 mm (mediana de 6 frente a 4.5)",
    labels={"nivel_lluvia": "Lluvia acumulada en la capital",
            "declaratorias": "Declaratorias de emergencia"},
    height=550,
)
fig3.update_layout(showlegend=False,           # la leyenda repite lo que ya dice el eje X
                   template="plotly_white", title_font_size=16, margin=dict(b=100))
fig3.add_annotation(text=FUENTE, xref="paper", yref="paper", x=0, y=-0.2,
                    showarrow=False, font=dict(size=10, color="gray"), xanchor="left")

## 4. Verificación rápida de valores extremos

Comprobamos que los valores más altos y más bajos de la tabla coinciden con lo que muestran los gráficos
y con lo que dicen sus títulos.

In [12]:
# Máximo de declaratorias y quiénes lo tienen (gráfico 1)
maximo = dataset["declaratorias"].max()
print("Máximo de declaratorias:", maximo)
print(dataset[dataset["declaratorias"] == maximo]["departamento"].tolist())

# Departamentos sin declaratorias (puntos en la base del gráfico 2)
print("\nSin declaratorias:")
print(dataset[dataset["declaratorias"] == 0]["departamento"].tolist())

# Extremos de lluvia (gráficos 1b y 2)
print("\nMás y menos lluvia:")
print(dataset.nlargest(1, "lluvia_total_mm")[["departamento", "lluvia_total_mm", "declaratorias"]].to_string(index=False))
print(dataset.nsmallest(1, "lluvia_total_mm")[["departamento", "lluvia_total_mm", "declaratorias"]].to_string(index=False))

# Medianas del boxplot (gráfico 3)
print("\nMediana de declaratorias por nivel de lluvia:")
print(dataset.groupby("nivel_lluvia", observed=True)["declaratorias"].median())

Máximo de declaratorias: 7
['Apurímac', 'Arequipa', 'Ayacucho', 'Cusco', 'Huancavelica']

Sin declaratorias:
['Callao', 'Lambayeque', 'Piura', 'Tumbes']

Más y menos lluvia:
departamento  lluvia_total_mm  declaratorias
      Loreto           1361.8              3
departamento  lluvia_total_mm  declaratorias
        Lima             12.2              6

Mediana de declaratorias por nivel de lluvia:
nivel_lluvia
Menos de 200 mm    3.0
200 a 700 mm       6.0
Más de 700 mm      4.5
Name: declaratorias, dtype: float64


In [13]:
# assert = "esto TIENE que ser verdad".

# Gráfico 1: cinco departamentos con 7 declaratorias
assert set(dataset[dataset["declaratorias"] == 7]["departamento"]) == {
    "Apurímac", "Arequipa", "Ayacucho", "Cusco", "Huancavelica"}

# Gráfico 1b: entre los 10 con más lluvia, solo Huancavelica tiene 7
top10_lluvia = dataset.nlargest(10, "lluvia_total_mm")
assert top10_lluvia[top10_lluvia["declaratorias"] == 7]["departamento"].tolist() == ["Huancavelica"]

# Gráfico 2: Loreto es el de más lluvia y tuvo 3 declaratorias
assert dataset.nlargest(1, "lluvia_total_mm")["departamento"].iloc[0] == "Loreto"
assert dataset.loc[dataset["departamento"] == "Loreto", "declaratorias"].iloc[0] == 3

# Gráfico 3: la mediana más alta es la del grupo de lluvia media, y vale 6
medianas = dataset.groupby("nivel_lluvia", observed=True)["declaratorias"].median()
assert medianas.idxmax() == "200 a 700 mm" and medianas.max() == 6

print("Los títulos coinciden con los datos ✔")

Los títulos coinciden con los datos ✔


## 5. Exportar un gráfico a HTML

Guardamos el boxplot como archivo HTML: se abre en cualquier navegador y conserva la interactividad, sin necesidad de Python.

In [14]:
# Paso 6: exportamos el gráfico 3 (boxplot) a un archivo HTML interactivo
fig3.write_html("grafico.html")
print("Gráfico exportado a grafico.html")

Gráfico exportado a grafico.html
